In [ ]:
import pandas as pd
import numpy as np

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Load dataset
df = pd.read_csv("online_retail.csv")

print("Original Shape:", df.shape)
df.head()

Original Shape: (399, 6)


,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
1,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
2,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
4,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom


In [2]:
# Remove missing values
df = df.dropna(subset=['Description', 'CustomerID'])

# Convert date
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Keep only required columns
df = df[['CustomerID', 'Description', 'InvoiceDate']]

print("After Cleaning:", df.shape)

After Cleaning: (399, 3)


In [ ]:
# Keeping a subset of data so it doesn't take 20 minutes to run Apriori


# Count product frequency
item_counts = df['Description'].value_counts()

# Keep top 50most frequent products
top_items = item_counts.head(50).index

df = df[df['Description'].isin(top_items)]

print("Remaining Unique Items:", df['Description'].nunique())

Remaining Unique Items: 50


In [4]:
# Create transaction identifier
df['TransactionID'] = (
    df['CustomerID'].astype(str) + "_" +
    df['InvoiceDate'].astype(str)
)

# Create baskets
baskets = df.groupby('TransactionID')['Description'].apply(list).tolist()

print("Total Transactions:", len(baskets))
print("Example Basket:", baskets[0])

Total Transactions: 34
Example Basket: ['VINTAGE HEADS AND TAILS CARD GAME', 'SET OF 3 COLOURED  FLYING DUCKS', 'ALARM CLOCK BAKELIKE GREEN', 'ALARM CLOCK BAKELIKE RED']


In [5]:
# Extract hour
df['Hour'] = df['InvoiceDate'].dt.hour

def get_time_period(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df['TimePeriod'] = df['Hour'].apply(get_time_period)

def create_baskets(time_period):
    temp = df[df['TimePeriod'] == time_period]
    return temp.groupby('TransactionID')['Description'].apply(list).tolist()

morning_baskets = create_baskets("Morning")
afternoon_baskets = create_baskets("Afternoon")
evening_baskets = create_baskets("Evening")
night_baskets = create_baskets("Night")

print("Morning:", len(morning_baskets))
print("Afternoon:", len(afternoon_baskets))
print("Evening:", len(evening_baskets))
print("Night:", len(night_baskets))

Morning: 34
Afternoon: 0
Evening: 0
Night: 0


In [6]:
def encode_transactions(transactions):
    
    items = sorted(set(item for basket in transactions for item in basket))
    encoded = []
    
    for basket in transactions:
        basket_set = set(basket)
        row = {}
        
        for item in items:
            row[item] = 1 if item in basket_set else 0
            
        encoded.append(row)
    
    return pd.DataFrame(encoded)

# Example encoding
custom_encoded = encode_transactions(morning_baskets)

print("Custom Encoding Shape:", custom_encoded.shape)
custom_encoded.head()

Custom Encoding Shape: (34, 50)


,60 TEATIME FAIRY CAKE CASES,ALARM CLOCK BAKELIKE GREEN,ALARM CLOCK BAKELIKE RED,ASSORTED COLOUR BIRD ORNAMENT,CLEAR DRAWER KNOB ACRYLIC EDWARDIAN,CREAM CUPID HEARTS COAT HANGER,DOORMAT FANCY FONT HOME SWEET HOME,EDWARDIAN PARASOL BLACK,EDWARDIAN PARASOL RED,GLASS STAR FROSTED T-LIGHT HOLDER,...,VINTAGE BILLBOARD DRINK ME MUG,VINTAGE BILLBOARD LOVE/HATE MUG,VINTAGE HEADS AND TAILS CARD GAME,WHITE HANGING HEART T-LIGHT HOLDER,WHITE METAL LANTERN,WOOD 2 DRAWER CABINET WHITE FINISH,WOOD S/3 CABINET ANT WHITE FINISH,WOODEN FRAME ANTIQUE WHITE,WOODEN PICTURE FRAME WHITE FINISH,YOU'RE CONFUSING ME METAL SIGN
0,0,1,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,0,1,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0


In [7]:
te = TransactionEncoder()
te_array = te.fit(morning_baskets).transform(morning_baskets)

library_encoded = pd.DataFrame(te_array, columns=te.columns_)

print("Library Encoding Shape:", library_encoded.shape)
library_encoded.head()

Library Encoding Shape: (34, 50)


,60 TEATIME FAIRY CAKE CASES,ALARM CLOCK BAKELIKE GREEN,ALARM CLOCK BAKELIKE RED,ASSORTED COLOUR BIRD ORNAMENT,CLEAR DRAWER KNOB ACRYLIC EDWARDIAN,CREAM CUPID HEARTS COAT HANGER,DOORMAT FANCY FONT HOME SWEET HOME,EDWARDIAN PARASOL BLACK,EDWARDIAN PARASOL RED,GLASS STAR FROSTED T-LIGHT HOLDER,...,VINTAGE BILLBOARD DRINK ME MUG,VINTAGE BILLBOARD LOVE/HATE MUG,VINTAGE HEADS AND TAILS CARD GAME,WHITE HANGING HEART T-LIGHT HOLDER,WHITE METAL LANTERN,WOOD 2 DRAWER CABINET WHITE FINISH,WOOD S/3 CABINET ANT WHITE FINISH,WOODEN FRAME ANTIQUE WHITE,WOODEN PICTURE FRAME WHITE FINISH,YOU'RE CONFUSING ME METAL SIGN
0,False,True,True,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
1,False,True,True,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False


In [8]:
print("Custom Shape:", custom_encoded.shape)
print("Library Shape:", library_encoded.shape)

print("Dimensions Equal:",
      custom_encoded.shape == library_encoded.shape)

Custom Shape: (34, 50)
Library Shape: (34, 50)
Dimensions Equal: True


In [9]:
time_groups = {
    "Morning": morning_baskets,
    "Afternoon": afternoon_baskets,
    "Evening": evening_baskets,
    "Night": night_baskets
}

encoded_groups = {}

for name, baskets in time_groups.items():
    print("Encoding:", name)
    
    te = TransactionEncoder()
    te_array = te.fit(baskets).transform(baskets)
    encoded_groups[name] = pd.DataFrame(te_array, columns=te.columns_)

Encoding: Morning
Encoding: Afternoon
Encoding: Evening
Encoding: Night


In [10]:
support_values = [0.05, 0.08]
results = {}

for support in support_values:
    
    print("\n==============================")
    print("SUPPORT =", support)
    print("==============================")
    
    for name, encoded_df in encoded_groups.items():
        
        freq = apriori(
            encoded_df,
            min_support=support,
            use_colnames=True,
            max_len=3,
            low_memory=True
        )
        
        if len(freq) > 0:
            max_len = freq['itemsets'].apply(lambda x: len(x)).max()
        else:
            max_len = 0
        
        print(f"{name}:")
        print("  Frequent Itemsets:", len(freq))
        print("  Largest Itemset Length:", max_len)
        
        results[(name, support)] = freq


SUPPORT = 0.05
Morning:
  Frequent Itemsets: 747
  Largest Itemset Length: 3
Afternoon:
  Frequent Itemsets: 0
  Largest Itemset Length: 0
Evening:
  Frequent Itemsets: 0
  Largest Itemset Length: 0
Night:
  Frequent Itemsets: 0
  Largest Itemset Length: 0

SUPPORT = 0.08
Morning:
  Frequent Itemsets: 713
  Largest Itemset Length: 3
Afternoon:
  Frequent Itemsets: 0
  Largest Itemset Length: 0
Evening:
  Frequent Itemsets: 0
  Largest Itemset Length: 0
Night:
  Frequent Itemsets: 0
  Largest Itemset Length: 0


In [11]:
def generate_rules(freq_items):
    
    if len(freq_items) == 0:
        return pd.DataFrame()
    
    rules = association_rules(
        freq_items,
        metric="confidence",
        min_threshold=0.6
    )
    
    rules = rules[rules['lift'] > 1]
    rules = rules.sort_values(by='lift', ascending=False)
    
    return rules

rule_results = {}

for key, freq in results.items():
    
    rules = generate_rules(freq)
    rule_results[key] = rules
    
    print(f"\nRules for {key}:")
    print("Number of Rules:", len(rules))


Rules for ('Morning', 0.05):
Number of Rules: 3523

Rules for ('Afternoon', 0.05):
Number of Rules: 0

Rules for ('Evening', 0.05):
Number of Rules: 0

Rules for ('Night', 0.05):
Number of Rules: 0

Rules for ('Morning', 0.08):
Number of Rules: 3485

Rules for ('Afternoon', 0.08):
Number of Rules: 0

Rules for ('Evening', 0.08):
Number of Rules: 0

Rules for ('Night', 0.08):
Number of Rules: 0


In [12]:
print("Strongest Rules by Lift\n")

for key, rules in rule_results.items():
    if len(rules) > 0:
        print(f"{key} → Highest Lift:", rules['lift'].max())

Strongest Rules by Lift

('Morning', 0.05) → Highest Lift: 17.0
('Morning', 0.08) → Highest Lift: 11.333333333333332


In [13]:
pair_tracker = {}

for key, freq in results.items():
    for itemset in freq['itemsets']:
        if len(itemset) == 2:
            pair = tuple(sorted(itemset))
            pair_tracker.setdefault(pair, []).append(key[0])

print("Pairs appearing in multiple time periods:\n")

for pair, periods in pair_tracker.items():
    if len(set(periods)) > 1:
        print(pair, "appears in:", set(periods))

Pairs appearing in multiple time periods:



Comparative Analysis

The strongest rules are identified using the highest Lift.

Item pairs appearing across multiple time periods indicate stable purchasing behavior.

Increasing support reduces:

Number of frequent itemsets

Number of rules

But increases rule strength.